# Agentes basados en conocimiento

Russell & Norvig, *Artificial Intelligence: A Modern Approach*, 4ª edición, capítulo 7.

*"En el que diseñamos agentes capaces de formar representaciones de un mundo complejo, utilizar un proceso de inferencia para derivar nuevas representaciones sobre el mundo y emplear estas nuevas representaciones para deducir qué hacer."*



In [1]:
#| include: false

import sys

# Los modulos comunes viven en 00-lib/, en la raiz del repo.
sys.path.append("../00-lib")


## Parte 1 — Definiciones

En la unidad anterior se le daba al agente solo un problema para optimizar, tenía los
estados, las acciones, los costos y el objetivo. "Sabía" cosas, pero de una
forma muy limitada e inflexible. 

En esta unidad se busca que el agente perciba una observación del mundo por vez y tenga que **inferir** el resto, en base a un conocimiento precargado. 

### 1.1 — La base de conocimiento

El componente central de un agente basado en conocimiento es su **base de
conocimiento** (*knowledge base*, **KB**), la cual consta de un conjunto de **sentencias**.

- **Sentencia**: cada una afirma algo sobre el mundo.
- **Lenguaje de representación de conocimiento**: el lenguaje en el que se escriben esas sentencias.
- **Axioma**: una sentencia que se toma como dada, que no se infiere de otras.
- **Conocimiento de fondo** (*background knowledge*): las sentencias de base del agente, antes de la primera percepción.

Además se definen dos operaciones: una para agregar sentencias y otra para consultar lo que se sabe.

```
TELL(KB, α)   agrega la sentencia α a la base
ASK(KB, α)    pregunta qué acción tomar
```

Las dos involucran **inferencia**: derivar sentencias nuevas a partir de las ya conocidas.


In [2]:
class KnowledgeBase:
    """Base de conocimiento abstracta. Para definir una concreta se heredan
    TELL y ASK."""

    def __init__(self, sentences=()):
        # El conocimiento de fondo: lo que la base ya sabe antes de la
        # primera percepcion.
        self.sentences = list(sentences)

    def TELL(self, sentence):
        """Agrega `sentence` a la base."""
        raise NotImplementedError

    def ASK(self, query):
        """Responde qué se sigue de lo que la base ya tiene."""
        raise NotImplementedError


Con esas dos operaciones ya se puede escribir el programa del agente. La idea es usar el mismo esqueleto para cualquier agente basado en conocimiento: recibe una
percepción y devuelve una acción.


In [3]:
def make_percept_sentence(percept, t):
    """La sentencia "percibi `percept` en el instante `t`"."""
    raise NotImplementedError


def make_action_query(t):
    """La consulta "¿que accion corresponde en el instante `t`?"."""
    raise NotImplementedError


def make_action_sentence(action, t):
    """La sentencia "hice `action` en el instante `t`"."""
    raise NotImplementedError


def kb_agent(percept, KB, t):
    """KB-AGENT(percept) retorna una acción."""
    KB.TELL(make_percept_sentence(percept, t))
    action = KB.ASK(make_action_query(t))
    KB.TELL(make_action_sentence(action, t))
    return action


Cada vez que se lo llama hace tres cosas: le cuenta a la base lo que
percibió, le pregunta qué acción conviene, y le cuenta que efectivamente la
hizo.

Las tres funciones `make_*` son la interfaz entre los sensores y actuadores
de un lado y la representación del otro: arman la sentencia que dice "percibí
esto en el instante t", la consulta "¿qué hago en el instante t?" y la
sentencia "hice esta acción en el instante t".


### 1.2 — Dos niveles de descripción

- **Nivel de conocimiento**: qué *sabe* el agente y cuáles son sus objetivos.
  Un taxi automático que tiene el objetivo de llevar un pasajero de San
  Francisco a Marin County, y que sabe que el Golden Gate es el único puente
  entre los dos lugares, va a cruzar el puente **porque sabe que eso logra su
  objetivo**. Para predecir su comportamiento no hizo falta hablar del
  programa.
- **Nivel de implementación**: cómo está hecho por dentro. Si el conocimiento
  geográfico está guardado como listas enlazadas, como mapas de píxeles o
  como señales propagándose por una red de neuronas es indistinto.

Y hay dos formas de construirlo.

- **Enfoque declarativo**: se arranca con la base vacía y el diseñador le va
  haciendo TELL de sentencias, una por una, hasta que el agente sepa operar
  en su entorno. Se le dice **qué** es verdad.
- **Enfoque procedimental**: los comportamientos deseados se escriben
  directamente como código. Se le dice **cómo** comportarse.

Hoy se entiende que un agente exitoso combina elementos de los ambos enfoques, y que el
conocimiento declarativo muchas veces se puede *compilar* a código
procedimental más eficiente.

Hay una tercera opción:
darle al agente mecanismos para **aprender** por su cuenta, generando
conocimiento general sobre el entorno a partir de una serie de percepciones.
Un agente que aprende puede ser completamente autónomo.


### 1.3 — Sintaxis, semántica y modelos

Un lenguaje de representación se define por dos cosas.

**Sintaxis**: qué sentencias están bien formadas. Por ejemplo, en aritmética, la
sintaxis es clara: `x + y = 4` es una sentencia bien formada, `x4y+ =` no lo
es.

**Semántica**: qué significan. Más precisamente, la semántica define la
**verdad** de cada sentencia respecto de cada **mundo posible**. La semántica
de la aritmética dice que `x + y = 4` es verdadera en un mundo donde `x` vale
2 e `y` vale 2, y falsa en un mundo donde `x` vale 1 e `y` vale 1. En
lógica estándar toda sentencia es verdadera o falsa.

En lugar de "mundo posible" se usa la palabra
**modelo**. Un mundo posible se puede pensar como un entorno real en el que
el agente puede estar o no; un modelo es una abstracción matemática que
simplemente fija el valor de verdad de toda sentencia relevante. Si la
sentencia α es verdadera en el modelo *m*, decimos que ***m* satisface α**, o
que *m* es un modelo de α.

```
M(α)    el conjunto de todos los modelos de α
```


### 1.4 — Implicación lógica

Ahora que tenemos una noción de verdad podemos hablar de razonamiento. La
relación clave es la **implicación lógica** (*entailment*): la idea de que
una sentencia *se sigue lógicamente* de otra.

```
α ⊨ β     la sentencia α implica la sentencia β
```

La definición formal es esta:

> α ⊨ β si y solo si, en todo modelo en el que α es verdadera, β también es
> verdadera.

Escrita con los conjuntos de modelos:

```
α ⊨ β   si y solo si   M(α) ⊆ M(β)
```

Veamos el mismo análisis en el **mundo del wumpus**, el entorno con el que
vamos a trabajar en la Parte 2: una cueva de 4×4 habitaciones, algunas con
pozos, donde vale la regla de que **hay brisa exactamente en las habitaciones
vecinas a un pozo**.

El agente arranca en `[1,1]`, no percibe nada, y se mueve a `[2,1]`, donde
percibe brisa. Su KB son las reglas del mundo más esas dos percepciones. La
pregunta es si hay pozo en `[1,2]`, en `[2,2]` o en `[3,1]`.

Tres habitaciones, cada una con pozo o sin pozo: 2³ = 8 modelos posibles. Los
enumeramos todos y nos quedamos con los que satisfacen la KB.


In [4]:
from itertools import product

# Un modelo le asigna un valor de verdad a cada simbolo. Aca los simbolos
# son "hay un pozo en tal habitacion", para las tres en cuestion.
habitaciones = ["[1,2]", "[2,2]", "[3,1]"]

modelos = [dict(zip(habitaciones, valores))
           for valores in product([False, True], repeat=3)]

len(modelos)


8

In [5]:
def KB(m):
    """¿Es verdadera la base de conocimiento del agente en el modelo `m`?

    Son las reglas del mundo junto con lo que el agente percibio. Las
    habitaciones [1,1] y [2,1] estan visitadas, asi que ya se sabe que no
    tienen pozo.
    """
    # No hay brisa en [1,1]: ninguna vecina de [1,1] tiene pozo. La unica
    # vecina en duda es [1,2].
    sin_brisa_en_1_1 = not m["[1,2]"]

    # Hay brisa en [2,1]: alguna vecina de [2,1] tiene pozo. Las vecinas en
    # duda son [2,2] y [3,1].
    brisa_en_2_1 = m["[2,2]"] or m["[3,1]"]

    return sin_brisa_en_1_1 and brisa_en_2_1


modelos_kb = [m for m in modelos if KB(m)]

print("".join(f"{h:>8}" for h in habitaciones), "  KB")
for m in modelos:
    fila = "".join(f"{'pozo' if m[h] else '-':>8}" for h in habitaciones)
    print(fila, "   *" if KB(m) else "")


   [1,2]   [2,2]   [3,1]   KB
       -       -       - 
       -       -    pozo    *
       -    pozo       -    *
       -    pozo    pozo    *
    pozo       -       - 
    pozo       -    pozo 
    pozo    pozo       - 
    pozo    pozo    pozo 


De los ocho modelos, la KB es verdadera en tres. Sobre esos tres preguntamos
por dos conclusiones posibles:

```
α1 = "no hay pozo en [1,2]"
α2 = "no hay pozo en [2,2]"
```


In [6]:
def alfa_1(m):
    return not m["[1,2]"]


def alfa_2(m):
    return not m["[2,2]"]


# α se sigue de la KB si es verdadera en TODO modelo donde la KB lo es.
print("KB ⊨ α1:", all(alfa_1(m) for m in modelos_kb))
print("KB ⊨ α2:", all(alfa_2(m) for m in modelos_kb))


KB ⊨ α1: True
KB ⊨ α2: False


In [7]:
#| echo: false
from figuras import ver_modelos

ver_modelos(modelos, habitaciones,
            {"KB": KB, "α1": alfa_1, "α2": alfa_2},
            visitadas={"[1,1]": "sin brisa", "[2,1]": "brisa"},
            agente="[2,1]")


En los tres modelos donde la KB es verdadera, α1 también lo es: **KB ⊨ α1**,
el agente puede concluir que en `[1,2]` no hay pozo. En cambio α2 es falsa en
alguno de esos tres, así que **KB ⊭ α2**. ¿Qué significa exactamente eso?


::: {.content-hidden}
Que el agente **no puede concluir que no haya** pozo en `[2,2]`. Y tampoco
puede concluir que lo haya: α2 es verdadera en uno de los tres modelos
y falsa en los otros dos. La información que tiene no alcanza para decidir, no que la
respuesta sea "no". Es la diferencia entre no saber y saber que no.
:::


El algoritmo que acabamos de correr —enumerar todos los modelos y verificar
que α es verdadera en todos aquellos donde la KB lo es— se llama
**verificación de modelos** (*model checking*), y es una forma de hacer
**inferencia lógica**: derivar conclusiones a partir de la definición de
implicación.

Si un algoritmo de inferencia *i* puede derivar α a partir de la KB, se
escribe:

```
KB ⊢i α     "α se deriva de KB por i"
```

La imagen del libro para separar las dos ideas: si el conjunto de todas las
consecuencias de la KB es un pajar y α es una aguja, la **implicación** es
que la aguja esté en el pajar, y la **inferencia** es encontrarla.


### 1.5 — Solidez y completitud

Las siguientes propiedades son deseables en un algoritmo de inferencia:

- **Solidez** (*soundness*), o que **preserve la verdad**: deriva únicamente
  sentencias implicadas. Un procedimiento no sólido hechos. La verificación de modelos, cuando se puede aplicar, es sólida.
- **Completitud** (*completeness*): puede derivar toda sentencia implicada. 

### 1.6 — Lógica proposicional

Lo visto hasta ahora puede aplicarse a cualquier esquema de lógica. Ahora tomamos un caso particular: la **lógica proposicional**.

#### 1.6.1 — Sintaxis

Las **sentencias atómicas** constan de un solo **símbolo proposicional**, que
representa una proposición que puede ser verdadera o falsa. Ej: `P`, `Q`, `R`, `W1,3`, `FacingEast`.

Las **sentencias complejas** se construyen a partir de sentencias más simples
usando **conectivos lógicos**. Son cinco:

| | conectivo | ejemplo | 
|---|---|---|
| `¬` | negación (*not*) | `¬W1,3` | 
| `∧` | conjunción (*and*) | `W1,3 ∧ P3,1` | 
| `∨` | disyunción (*or*) | `(W1,3 ∧ P3,1) ∨ W2,2` |
| `⇒` | implicación (*implies*) | `(W1,3 ∧ P3,1) ⇒ ¬W2,2` |
| `⇔` | bicondicional (*if and only if*) | `W1,3 ⇔ ¬W2,2` | 

#### 1.6.2 — Semántica

La semántica define cómo determinar el valor de verdad de una sentencia
respecto de un modelo. En lógica proposicional un **modelo** es simplemente
una asignación de un **valor de verdad** —verdadero o falso— a cada símbolo
proposicional. Si las sentencias de la base usan los símbolos `P1,2`, `P2,2` y
`P3,1`, un modelo posible es:

```
m1 = {P1,2 = falso,  P2,2 = falso,  P3,1 = verdadero}
```

Con tres símbolos hay 2³ = 8 modelos posibles, con *n* símbolos hay 2ⁿ. 

Entonces, para las sentencias complejas hay cinco reglas, una por conectivo, que valen
para cualesquiera subsentencias `P` y `Q` en cualquier modelo *m*:

- `¬P` es verdadera si y solo si `P` es falsa en *m*.
- `P ∧ Q` es verdadera si y solo si `P` y `Q` son las dos verdaderas en *m*.
- `P ∨ Q` es verdadera si y solo si `P` o `Q` es verdadera en *m*.
- `P ⇒ Q` es verdadera salvo que `P` sea verdadera y `Q` falsa en *m*.
- `P ⇔ Q` es verdadera si y solo si `P` y `Q` son las dos verdaderas o las
  dos falsas en *m*.

Escritas como funciones de Python:

In [8]:
def NOT(P):
    return not P


def AND(P, Q):
    return P and Q


def OR(P, Q):
    return P or Q


def IMPLIES(P, Q):
    # Falsa solo cuando P es verdadera y Q es falsa.
    return (not P) or Q


def IFF(P, Q):
    return P == Q


Las reglas se pueden expresar también como **tablas de verdad**, que dan el
valor de la sentencia compleja para cada asignación posible a sus partes. Con
las cinco funciones de arriba, la tabla de verdad de los cinco conectivos
sale sola:


In [9]:
print(f"{'P':>7}{'Q':>7}{'¬P':>7}{'P∧Q':>7}{'P∨Q':>7}{'P⇒Q':>7}{'P⇔Q':>7}")

for P, Q in product([False, True], repeat=2):
    fila = [P, Q, NOT(P), AND(P, Q), OR(P, Q), IMPLIES(P, Q), IFF(P, Q)]
    print("".join(f"{str(v).lower():>7}" for v in fila))


      P      Q     ¬P    P∧Q    P∨Q    P⇒Q    P⇔Q
  false  false   true  false  false   true   true
  false   true   true  false   true   true  false
   true  false  false  false   true  false  false
   true   true  false   true   true   true   true


#### 1.6.3 — Escribiendo la base de conocimiento

Con esto ya podemos escribir en el lenguaje la KB que en 1.4 habíamos escrito
como una función de Python. Para cada habitación `[x,y]` del mundo del wumpus
necesitamos estos símbolos:

```
Px,y    es verdadero si hay un pozo en [x,y]
Wx,y    es verdadero si hay un wumpus en [x,y]
Bx,y    es verdadero si hay brisa en [x,y]
Sx,y    es verdadero si hay hedor en [x,y]
Lx,y    es verdadero si el agente está en [x,y]
```

Y las sentencias:

```
R1:  ¬P1,1
R2:  B1,1 ⇔ (P1,2 ∨ P2,1)
R3:  B2,1 ⇔ (P1,1 ∨ P2,2 ∨ P3,1)
R4:  ¬B1,1
R5:  B2,1
```


R1, R2 y R3 son verdaderas en todos los mundos del wumpus: son los
**axiomas** del mundo, el conocimiento de fondo del agente. 

R4 y R5 son de esta interacción en particular que ingresaron a la KB por TELL.


### 1.7 — Propiedades y reglas de inferencia

Veamos las siguientes cuatro propiedades que se apoyan en la implicación.

**Equivalencia lógica**: dos sentencias α y β son lógicamente equivalentes si
son verdaderas en el mismo conjunto de modelos. Se escribe `α ≡ β`, y
equivale a decir que cada una implica a la otra: `α ≡ β` si y solo si `α ⊨ β`
y `β ⊨ α`. Algunas equivalencias lógicas son:

```
¬(¬α)          ≡  α                             doble negación
(α ⇒ β)        ≡  (¬β ⇒ ¬α)                     contraposición
(α ⇒ β)        ≡  (¬α ∨ β)                      eliminación de la implicación
(α ⇔ β)        ≡  ((α ⇒ β) ∧ (β ⇒ α))           eliminación del bicondicional
¬(α ∧ β)       ≡  (¬α ∨ ¬β)                     De Morgan
¬(α ∨ β)       ≡  (¬α ∧ ¬β)                     De Morgan
(α ∧ (β ∨ γ))  ≡  ((α ∧ β) ∨ (α ∧ γ))           distributividad de ∧ sobre ∨
(α ∨ (β ∧ γ))  ≡  ((α ∨ β) ∧ (α ∨ γ))           distributividad de ∨ sobre ∧
```

**Validez**: una sentencia es válida si es verdadera en **todos** los
modelos. `P ∨ ¬P` es válida. También se las llama **tautologías**: son
necesariamente verdaderas. De acá sale el **teorema de deducción**:

> para cualesquiera α y β, `α ⊨ β` si y solo si la sentencia `(α ⇒ β)` es
> válida.

**Satisfacibilidad**: una sentencia es satisfacible si es verdadera en
**algún** modelo. La KB de 1.4 es satisfacible: hay tres modelos que la
satisfacen. La validez y satisfacibilidad están conectadas: α es válida si y
solo si `¬α` es insatisfacible. Y de ahí sale el resultado que vamos a usar
para hacer inferencia:

> `α ⊨ β` si y solo si la sentencia `(α ∧ ¬β)` es insatisfacible.

Probar β a partir de α mostrando que `(α ∧ ¬β)` es insatisfacible es
exactamente la técnica del *reductio ad absurdum*: se supone que β es falsa y
se muestra que eso lleva a una contradicción. También se la llama prueba por
**refutación** o por contradicción.

EJ: 
Si llueve entonces la calle está mojada
`(L→M)∧L ⊨ M`
`α∧¬β` ->  `[(L→M)∧L]∧¬M`

**Monotonicidad**: el conjunto de sentencias implicadas solo puede **crecer**
a medida que se agrega información a la base.

```
si   KB ⊨ α    entonces   KB ∧ β ⊨ α
```

Agregar β puede permitir sacar conclusiones nuevas, pero no puede invalidar
ninguna conclusión α que ya se había sacado.

#### 1.7.1 — Pruebas

Para no enumerar todos los modelos posibles se aplican **reglas de inferencia**
directamente a las sentencias de la base, encadenándolas para construir una
**prueba**.

Una regla muy conocida es **Modus Ponens**: dadas una implicación y su
premisa, se puede inferir la conclusión.

```
α ⇒ β,   α
───────────
     β
```

Otra es la **eliminación de la conjunción**: de una conjunción se puede
inferir cualquiera de sus partes.

```
α ∧ β
──────
   α
```

Cada una de las equivalencias lógicas de más arriba también sirve como regla
de inferencia. Y buscar una prueba es, otra vez, un problema de búsqueda como
los de la unidad 1, así que cualquiera de aquellos algoritmos sirve para
encontrarla:

- **INITIAL**: la base de conocimiento inicial.
- **ACTIONS**: todas las reglas de inferencia aplicables a las sentencias que
  hay.
- **RESULT**: agregar a la base la sentencia de abajo de la regla aplicada.
- **GOAL**: un estado que contenga la sentencia que se quiere probar.

Las reglas de esta sección son sólidas, pero un conjunto de reglas puede ser
**incompleto**: si sacáramos, por ejemplo, la eliminación del bicondicional,
hay pruebas que dejarían de existir. Existe una única regla de inferencia, la
**resolución**, que acoplada a cualquier algoritmo de búsqueda completo da un
procedimiento de inferencia completo. Es lo que vamos a ver en la Parte 3.


### 1.8 — Cláusulas, FNC y cláusulas de Horn

La regla de resolución se aplica solo a **cláusulas**: disyunciones de
literales, como `(P1,1 ∨ P3,1)`. 

Toda sentencia de la lógica proposicional es lógicamente equivalente a una conjunción de
cláusulas. Una sentencia escrita de esa forma está en **forma normal
conjuntiva** (**FNC**, o *CNF*). El procedimiento para convertirla usa las
equivalencias de 1.7. Por ejemplo, aplicado sobre `R2`:

```
R2:  B1,1 ⇔ (P1,2 ∨ P2,1)

1. eliminar ⇔       (B1,1 ⇒ (P1,2 ∨ P2,1)) ∧ ((P1,2 ∨ P2,1) ⇒ B1,1)
2. eliminar ⇒       (¬B1,1 ∨ P1,2 ∨ P2,1) ∧ (¬(P1,2 ∨ P2,1) ∨ B1,1)
3. ¬ hacia adentro  (¬B1,1 ∨ P1,2 ∨ P2,1) ∧ ((¬P1,2 ∧ ¬P2,1) ∨ B1,1)
4. distribuir ∨     (¬B1,1 ∨ P1,2 ∨ P2,1) ∧ (¬P1,2 ∨ B1,1) ∧ (¬P2,1 ∨ B1,1)
```

La sentencia original quedó como una conjunción de tres cláusulas.

Dentro de las cláusulas hay dos familias restringidas que habilitan métodos de inferencia más eficientes:

- **Cláusula definida**: una disyunción de literales de los cuales
  **exactamente uno es positivo**. `(¬L1,1 ∨ ¬Breeze ∨ B1,1)` es una cláusula
  definida; `(¬B1,1 ∨ P1,2 ∨ P2,1)` no lo es, porque tiene dos positivos.

- **Cláusula de Horn**: una disyunción de literales de los cuales **a lo sumo
  uno es positivo**. Toda cláusula definida es de Horn. Las que no tienen
  ningún literal positivo se llaman **cláusulas objetivo**. Las cláusulas de
  Horn son cerradas bajo resolución: al resolver dos cláusulas de Horn se
  obtiene otra cláusula de Horn.

Las bases formadas solo por cláusulas definidas son interesantes por tres
razones.

**1.** Toda cláusula definida se puede escribir como una **implicación** cuya
premisa es una conjunción de literales positivos y cuya conclusión es un
único literal positivo. La cláusula de recién se escribe:

```
(¬L1,1 ∨ ¬Breeze ∨ B1,1)      ≡      (L1,1 ∧ Breeze) ⇒ B1,1
```

y así se lee mucho mejor: si el agente está en `[1,1]` y percibe brisa,
entonces hay brisa en `[1,1]`. En forma de Horn, la premisa se llama
**cuerpo** y la conclusión, **cabeza**. Una sentencia formada por un único
literal positivo, como `L1,1`, se llama **hecho**: también se puede escribir
en forma de implicación, como `True ⇒ L1,1`, pero es más simple escribir
directamente `L1,1`.

**2.** La inferencia con cláusulas de Horn se puede hacer con los algoritmos
de **encadenamiento hacia adelante** (*forward chaining*) y **hacia atrás**
(*backward chaining*). Los dos son naturales: los pasos son obvios y fáciles
de seguir para una persona, a diferencia de lo que pasa con la resolución.
Este tipo de inferencia es la base de la **programación lógica**.

**3.** Decidir la implicación con cláusulas de Horn lleva un tiempo **lineal**
en el tamaño de la base de conocimiento. 


***
## Parte 2 — Encadenamiento hacia adelante y hacia atrás

En 1.8 quedó dicho que las bases formadas por cláusulas definidas se pueden
tratar con dos algoritmos que no enumeran modelos ni convierten nada a FNC:
el **encadenamiento hacia adelante** y el **encadenamiento hacia atrás**. Los
dos son sólidos y completos para ese tipo de base.

### 2.1 — Una base de cláusulas definidas

Una cláusula definida tiene exactamente un literal positivo, así que siempre
se puede escribir como una implicación: el **cuerpo** es una conjunción de
símbolos y la **cabeza** es un símbolo solo. La representamos con esas dos
partes.


In [10]:
class Clause:
    """Una clausula definida, escrita como implicacion.

    PREMISE es el cuerpo: la lista de simbolos que se conjugan.
    CONCLUSION es la cabeza: el unico simbolo positivo.

    Un hecho es una clausula de cuerpo vacio: `True => A` se escribe `A`.
    """

    def __init__(self, PREMISE, CONCLUSION):
        self.PREMISE = PREMISE
        self.CONCLUSION = CONCLUSION

    def __repr__(self):
        if not self.PREMISE:
            return self.CONCLUSION
        return " ∧ ".join(self.PREMISE) + " ⇒ " + self.CONCLUSION


La base con la que vamos a trabajar es la del libro. Son cinco reglas y dos
hechos, y la consulta va a ser el símbolo `Q`.


In [11]:
clausulas = [
    Clause(["P"], "Q"),
    Clause(["L", "M"], "P"),
    Clause(["B", "L"], "M"),
    Clause(["A", "P"], "L"),
    Clause(["A", "B"], "L"),
    Clause([], "A"),
    Clause([], "B"),
]

for c in clausulas:
    print(c)


P ⇒ Q
L ∧ M ⇒ P
B ∧ L ⇒ M
A ∧ P ⇒ L
A ∧ B ⇒ L
A
B


Esta base también se puede dibujar como un **grafo AND–OR**. Cada símbolo es un nodo y cada cláusula,
una arista que va del cuerpo a la cabeza:

- varias aristas unidas por un arco son una **conjunción**: hay que probar
  todas. Es lo que pasa con `L` y `M` para llegar a `P`.
- varias aristas sin arco son una **disyunción**: alcanza con probar alguna.
  Es lo que pasa con `L`, al que se llega por dos cláusulas distintas.

![La misma base: a la izquierda las cláusulas, a la derecha el grafo.
Russell & Norvig, figura 7.16.](figuras/7.16-grafo-and-or.png)


### 2.2 — Encadenamiento hacia adelante

Arranca de los hechos —los literales positivos que ya están en la base— y
propaga hacia arriba. Si se conocen todas las premisas de una implicación, se
agrega su conclusión al conjunto de hechos conocidos. Sigue hasta que aparece
la consulta o hasta que no se puede inferir nada más.

Lo que hace que corra en tiempo lineal es la tabla `count`: para cada
cláusula guarda cuántos símbolos de su cuerpo faltan probar. Cada vez que se
procesa un símbolo nuevo se les resta uno a las cláusulas donde aparece, y
cuando un contador llega a cero, esa cláusula ya se puede disparar. Así nunca
hay que revisar una cláusula entera de nuevo.


In [12]:
from collections import deque


def pl_fc_entails(KB, q):
    """PL-FC-ENTAILS?(KB, q) returns true or false.

    `KB` es una lista de clausulas definidas y `q` un simbolo.
    """
    # Cuantos simbolos del cuerpo de cada clausula faltan probar.
    count = {c: len(c.PREMISE) for c in KB}

    # Que simbolos ya se procesaron, para no repetir trabajo ni entrar en
    # bucle con cosas como P ⇒ Q y Q ⇒ P.
    inferred = {}

    # Los simbolos que se saben verdaderos pero todavia no se procesaron:
    # al principio, los hechos.
    queue = deque(c.CONCLUSION for c in KB if not c.PREMISE)

    while queue:
        p = queue.popleft()
        if p == q:
            return True
        if not inferred.get(p, False):
            inferred[p] = True
            for c in KB:
                if p in c.PREMISE:
                    count[c] -= 1
                    if count[c] == 0:
                        queue.append(c.CONCLUSION)
    return False


In [13]:
print("¿la base implica Q?", pl_fc_entails(clausulas, "Q"))
print("¿la base implica Z?", pl_fc_entails(clausulas, "Z"))


¿la base implica Q? True
¿la base implica Z? False


Paso a paso, para ver los contadores bajando:


In [14]:
#| include: false
def pl_fc_entails_v(KB, q):
    """Igual que pl_fc_entails, narrando cada paso."""
    count = {c: len(c.PREMISE) for c in KB}
    inferred = {}
    queue = deque(c.CONCLUSION for c in KB if not c.PREMISE)

    print("hechos conocidos:", ", ".join(queue))
    while queue:
        p = queue.popleft()
        if p == q:
            print(f"saco {p}   es la consulta")
            return True
        print(f"saco {p}")
        if not inferred.get(p, False):
            inferred[p] = True
            for c in KB:
                if p in c.PREMISE:
                    count[c] -= 1
                    disparo = f"→ {c.CONCLUSION}" if count[c] == 0 else ""
                    print(f"     {str(c):<14} le faltan {count[c]}  {disparo}".rstrip())
                    if count[c] == 0:
                        queue.append(c.CONCLUSION)
    return False


In [15]:
#| echo: false
_ = pl_fc_entails_v(clausulas, "Q")


hechos conocidos: A, B
saco A
     A ∧ P ⇒ L      le faltan 1
     A ∧ B ⇒ L      le faltan 1
saco B
     B ∧ L ⇒ M      le faltan 1
     A ∧ B ⇒ L      le faltan 0  → L
saco L
     L ∧ M ⇒ P      le faltan 1
     B ∧ L ⇒ M      le faltan 0  → M
saco M
     L ∧ M ⇒ P      le faltan 0  → P
saco P
     P ⇒ Q          le faltan 0  → Q
     A ∧ P ⇒ L      le faltan 0  → L
saco Q   es la consulta


El algoritmo es **sólido**: cada paso es una aplicación de Modus Ponens. Y es
**completo**: toda sentencia atómica implicada por la base se deriva. La
forma de verlo es mirar el estado final de la tabla `inferred`, cuando ya no
se puede inferir nada más. Esa tabla, leída como un modelo —verdadero para
cada símbolo inferido, falso para todos los demás—, es un modelo de la base:
si alguna cláusula `a1 ∧ ... ∧ ak ⇒ b` fuera falsa en él, su cuerpo sería
verdadero y su cabeza falsa, y entonces el algoritmo no habría terminado.
Como toda sentencia implicada es verdadera en todos los modelos de la base, y
en particular en este, toda sentencia implicada termina inferida.

El encadenamiento hacia adelante es un ejemplo de **razonamiento dirigido por
los datos** (*data-driven*): el foco arranca en lo que se sabe. Un agente
puede usarlo sin tener ninguna consulta en mente, para ir procesando las
percepciones a medida que llegan. El riesgo es el mismo que en las personas:
de cada dato nuevo se siguen infinitas consecuencias irrelevantes, y hay que
mantener el proceso bajo control para no ahogarse en ellas.


### 2.3 — Encadenamiento hacia atrás

Va al revés: arranca en la consulta. Si `q` ya se sabe verdadero, no hay nada
que hacer. Si no, busca en la base las implicaciones cuya **conclusión** sea
`q`, y para cada una intenta probar todas sus premisas, otra vez hacia atrás.
Baja por el grafo hasta llegar a hechos conocidos.

Las dos condiciones de corte importan. Si un subobjetivo ya está en la cadena
de objetivos que se están intentando probar más arriba, hay un **ciclo** y esa
rama se abandona. Y para no repetir trabajo, conviene recordar los
subobjetivos que ya se probaron y los que ya fallaron; la versión de acá no lo
hace, y en la traza se va a ver el costo.


In [16]:
def pl_bc_entails(KB, q, objetivos=()):
    """¿Se sigue el simbolo `q` de KB? Trabaja hacia atras, desde la consulta.

    `objetivos` son los simbolos que se estan intentando probar mas arriba en
    la cadena: si `q` vuelve a aparecer ahi, es un ciclo.
    """
    if q in objetivos:
        return False
    for c in KB:
        if c.CONCLUSION == q:
            # Un hecho tiene el cuerpo vacio, y all([]) es True.
            if all(pl_bc_entails(KB, p, objetivos + (q,)) for p in c.PREMISE):
                return True
    return False


In [17]:
print("¿la base implica Q?", pl_bc_entails(clausulas, "Q"))
print("¿la base implica Z?", pl_bc_entails(clausulas, "Z"))


¿la base implica Q? True
¿la base implica Z? False


La traza es directamente el grafo AND–OR, leído de arriba hacia abajo: cada
sangría es un nivel más abajo, los símbolos de una misma cláusula son la
conjunción y las cláusulas alternativas son la disyunción.


In [18]:
#| include: false
def pl_bc_entails_v(KB, q, objetivos=(), nivel=0):
    """Igual que pl_bc_entails, narrando cada paso."""
    sangria = "    " * nivel
    if q in objetivos:
        print(f"{sangria}¿{q}?  ya se está probando más arriba: ciclo")
        return False
    for c in KB:
        if c.CONCLUSION == q:
            if not c.PREMISE:
                print(f"{sangria}¿{q}?  sí, es un hecho")
                return True
            print(f"{sangria}¿{q}?  pruebo con  {c}")
            if all(pl_bc_entails_v(KB, p, objetivos + (q,), nivel + 1)
                   for p in c.PREMISE):
                print(f"{sangria}{q} probado")
                return True
    print(f"{sangria}{q} no se pudo probar")
    return False


In [19]:
#| echo: false
_ = pl_bc_entails_v(clausulas, "Q")


¿Q?  pruebo con  P ⇒ Q
    ¿P?  pruebo con  L ∧ M ⇒ P
        ¿L?  pruebo con  A ∧ P ⇒ L
            ¿A?  sí, es un hecho
            ¿P?  ya se está probando más arriba: ciclo
        ¿L?  pruebo con  A ∧ B ⇒ L
            ¿A?  sí, es un hecho
            ¿B?  sí, es un hecho
        L probado
        ¿M?  pruebo con  B ∧ L ⇒ M
            ¿B?  sí, es un hecho
            ¿L?  pruebo con  A ∧ P ⇒ L
                ¿A?  sí, es un hecho
                ¿P?  ya se está probando más arriba: ciclo
            ¿L?  pruebo con  A ∧ B ⇒ L
                ¿A?  sí, es un hecho
                ¿B?  sí, es un hecho
            L probado
        M probado
    P probado
Q probado


Se ve todo lo que hay para ver. La primera cláusula que prueba para `L` es
`A ∧ P ⇒ L`, y `P` ya está más arriba en la cadena: ciclo, se abandona la
rama y pasa a `A ∧ B ⇒ L`, que sí cierra contra dos hechos. Y `L` se prueba
dos veces, una para `P` y otra para `M`, porque esta versión no recuerda lo
que ya probó. Una implementación que memorice los subobjetivos resueltos corre
en tiempo lineal.

El encadenamiento hacia atrás es una forma de **razonamiento dirigido por el
objetivo** (*goal-directed*). Sirve para contestar preguntas puntuales, del
estilo de "¿qué hago ahora?" o "¿dónde dejé las llaves?". Y su costo suele
ser **mucho menor que lineal** en el tamaño de la base, porque toca únicamente
los hechos relevantes: acá nunca miró la cláusula `B ∧ L ⇒ M` hasta que
necesitó `M`, y si la base tuviera mil cláusulas más sobre otros símbolos, no
las habría mirado nunca.

Es, otra vez, un problema de búsqueda de los de la unidad 1: el libro señala
que el algoritmo es esencialmente la búsqueda sobre un grafo AND–OR.

### 2.4 — Los dos, comparados

| | hacia adelante | hacia atrás |
|---|---|---|
| arranca en | los hechos conocidos | la consulta |
| razonamiento | dirigido por los datos | dirigido por el objetivo |
| infiere | todo lo que se sigue de la base | solo lo que hace falta |
| costo | lineal en el tamaño de la base | a menudo mucho menor que lineal |
| le sirve a un agente para | ir procesando las percepciones a medida que llegan | contestar "¿qué hago ahora?" |

El encadenamiento hacia adelante infirió `L`, `M` y `P` sin que nadie se lo
pidiera. ¿Es trabajo desperdiciado?


::: {.content-hidden}
Depende de para qué. Si la pregunta es una sola y concreta, sí: el hacia atrás
llegó a la misma respuesta tocando menos cláusulas. Pero un agente que percibe
el mundo no tiene una sola pregunta, tiene una percepción nueva por turno; ahí
conviene ir manteniendo al día todo lo que se sigue de lo percibido, porque
esas conclusiones se van a necesitar. Los dos algoritmos son sólidos y
completos: la diferencia no es qué se puede probar, sino qué trabajo se hace
para probarlo.
:::


***
## Parte 3 — El mundo del wumpus

Los dos algoritmos de la Parte 2 son baratos, pero solo sirven para bases de
cláusulas definidas, y el mundo del wumpus no lo es.

La regla central del mundo es que hay brisa en una casilla si y solo si alguna
vecina tiene un pozo. Escrita para `[1,1]`, es la `R2` de 1.6.3:

```
B1,1 ⇔ (P1,2 ∨ P2,1)
```

y en FNC, como vimos en 1.8, da tres cláusulas:

`(¬B1,1 ∨ P1,2 ∨ P2,1) ∧ (¬P1,2 ∨ B1,1) ∧ (¬P2,1 ∨ B1,1)`
 
tomando:

`(¬B1,1 ∨ P1,2 ∨ P2,1)`

Tiene **dos literales positivos**. No es una cláusula
definida ni de Horn, así que el encadenamiento no la puede usar.

### 3.1 — El entorno

La descripción PEAS, como en la unidad 1:

- **Medida de rendimiento**: +1000 por salir de la cueva con el oro, −1000 por
  caer en un pozo o ser comido por el wumpus, −1 por cada acción y −10 por
  usar la flecha.
- **Entorno**: una cueva de 4×4 habitaciones con paredes alrededor. El agente
  arranca en `[1,1]` mirando al este. El oro y el wumpus están en casillas
  cualesquiera menos la de arranque, y cada casilla que no sea la de arranque
  puede tener un pozo con probabilidad 0,2.
- **Actuadores**: `Forward`, `TurnLeft`, `TurnRight`, `Grab`, `Shoot` y
  `Climb`.
- **Sensores**: cinco bits. **hedor** en las casillas vecinas al wumpus,
  **brisa** en las vecinas a un pozo, **brillo** en la casilla del oro,
  **golpe** al chocar contra una pared y **grito**, audible desde cualquier
  lado, cuando el wumpus muere.

El entorno es determinista, discreto, estático y de un solo agente. Pero es
**parcialmente observable**: el agente no ve dónde están los pozos ni el
wumpus. 



In [20]:
def vecinas(c, n=4):
    """Las casillas adyacentes a `c` que caen adentro de la cueva."""
    x, y = c
    return [v for v in ((x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1))
            if 1 <= v[0] <= n and 1 <= v[1] <= n]


class MundoWumpus:
    """La cueva de verdad. El agente no ve nada de esto: solo la percibe."""

    def __init__(self, pozos, wumpus, oro, n=4):
        self.n = n
        self.pozos = set(pozos)
        self.wumpus = wumpus
        self.oro = oro

    def vecinas(self, c):
        return vecinas(c, self.n)

    def PERCEPT(self, c, tiene_oro=False):
        """Los cinco sensores, parado en la casilla `c`."""
        return {
            "hedor":  self.wumpus in self.vecinas(c),
            "brisa":  any(p in self.pozos for p in self.vecinas(c)),
            "brillo": c == self.oro and not tiene_oro,
            # Sin orientacion no hay choques, y sin flecha no hay gritos.
            "golpe":  False,
            "grito":  False,
        }

    def es_mortal(self, c):
        return c in self.pozos or c == self.wumpus


La cueva es la de la figura 7.2 del libro: pozos en `[3,1]`, `[3,3]` y
`[4,4]`, el wumpus en `[1,3]` y el oro en `[2,3]`.


In [21]:
mundo = MundoWumpus(pozos=[(3, 1), (3, 3), (4, 4)], wumpus=(1, 3), oro=(2, 3))

for c in [(1, 1), (2, 1), (1, 2), (2, 3)]:
    percibido = [s for s, hay in mundo.PERCEPT(c).items() if hay]
    print(f"en {c} se percibe:", ", ".join(percibido) or "nada")


en (1, 1) se percibe: nada
en (2, 1) se percibe: brisa
en (1, 2) se percibe: hedor
en (2, 3) se percibe: hedor, brisa, brillo


### 3.2 — La física del mundo

Los símbolos son los de 1.6.3, para cada casilla `[x,y]`:

```
Px,y    hay un pozo en [x,y]
Wx,y    hay un wumpus en [x,y]
Bx,y    hay brisa en [x,y]
Sx,y    hay hedor en [x,y]
OKx,y   es seguro entrar a [x,y]
```

Y estos son los axiomas. Valen en todos los mundos del wumpus y en todo
momento —por eso se los llama **atemporales**—, así que el agente los tiene
antes de dar el primer paso: son su conocimiento de fondo.

```
¬P1,1                                      la casilla de arranque no tiene pozo
¬W1,1                                      ni wumpus
Bx,y ⇔ (Pu ∨ Pv ∨ …)                       hay brisa si y solo si una vecina
Sx,y ⇔ (Wu ∨ Wv ∨ …)                       tiene pozo; idem hedor y wumpus
W1,1 ∨ W1,2 ∨ … ∨ W4,4                     hay al menos un wumpus
¬Wa ∨ ¬Wb   para cada par de casillas      y a lo sumo uno
OKx,y ⇔ ¬Px,y ∧ ¬(Wx,y ∧ WumpusAlive)      seguro es sin pozo y sin wumpus vivo
```

A los axiomas se les suman las percepciones, una por instante: `¬B1,1` en el
momento 0, `B2,1` en el momento 1, y así.

### 3.3 — Las sentencias

Una sentencia atómica es un `str` con el nombre del
símbolo, y una compuesta es una tupla que arranca con el conectivo.


In [ ]:
def Neg(a):         return ("¬", a)
def And(*ss):       return ("∧",) + ss
def Or(*ss):        return ("∨",) + ss
def Implies(a, b):  return ("⇒", a, b)
def Iff(a, b):      return ("⇔", a, b)


def escribir(s):
    """La sentencia `s`"""
    if isinstance(s, str):
        return s
    conectivo, *partes = s
    if conectivo == "¬":
        return "¬" + escribir(partes[0])
    if conectivo in ("∧", "∨"):
        return "(" + f" {conectivo} ".join(escribir(p) for p in partes) + ")"
    return f"({escribir(partes[0])} {conectivo} {escribir(partes[1])})"


print(escribir(Iff("B1,1", Or("P1,2", "P2,1"))))


(B1,1 ⇔ (P1,2 ∨ P2,1))


Falta la semántica: el valor de verdad de una sentencia en un modelo. Son las
cinco reglas de 1.6.2, aplicadas recursivamente hasta llegar a los símbolos.


In [23]:
def simbolos(s):
    """Los simbolos proposicionales que aparecen en la sentencia `s`."""
    if isinstance(s, str):
        return {s}
    return set().union(*(simbolos(p) for p in s[1:]))


def pl_true(s, model):
    """PL-TRUE?(s, m): el valor de verdad de `s` en el modelo `m`.

    Un modelo es un dict que le da verdadero o falso a cada simbolo.
    """
    if isinstance(s, str):
        return model[s]
    conectivo, *partes = s
    if conectivo == "¬":
        return NOT(pl_true(partes[0], model))
    # `all` y `any` son AND y OR, pero de a muchos en vez de a dos.
    if conectivo == "∧":
        return all(pl_true(p, model) for p in partes)
    if conectivo == "∨":
        return any(pl_true(p, model) for p in partes)
    if conectivo == "⇒":
        return IMPLIES(pl_true(partes[0], model), pl_true(partes[1], model))
    if conectivo == "⇔":
        return IFF(pl_true(partes[0], model), pl_true(partes[1], model))
    raise ValueError(f"conectivo desconocido: {conectivo}")


### 3.4 — Verificación de modelos

Con la semántica escrita, el algoritmo de inferencia sale solo: es la
definición de implicación de 1.4, hecha programa. Enumerar todos los modelos y
comprobar que α es verdadera en todos aquellos donde la KB lo es.

`tt_check_all` recorre los símbolos de a uno, probando las dos ramas —el
símbolo en verdadero y en falso—, y cuando ya no quedan símbolos el modelo
está completo y se lo evalúa.


In [24]:
def tt_entails(KB, alpha, symbols=None):
    """TT-ENTAILS?(KB, α) returns true or false.

    `symbols` permite enumerar solo algunos simbolos, que es lo que vamos a
    necesitar mas abajo. Por defecto, todos los que aparecen.
    """
    if symbols is None:
        symbols = sorted(simbolos(KB) | simbolos(alpha))
    return tt_check_all(KB, alpha, symbols, {})


def tt_check_all(KB, alpha, symbols, model):
    if not symbols:
        if pl_true(KB, model):
            return pl_true(alpha, model)
        # Donde la KB es falsa no hay nada que exigirle a α.
        return True
    P, *resto = symbols
    return (tt_check_all(KB, alpha, resto, model | {P: True})
            and tt_check_all(KB, alpha, resto, model | {P: False}))


Probémoslo con la situación de 1.4, ahora escrita en el lenguaje: el agente
estuvo en `[1,1]` sin percibir brisa y en `[2,1]` percibiendo brisa.


In [25]:
R1 = Neg("P1,1")
R2 = Iff("B1,1", Or("P1,2", "P2,1"))
R3 = Iff("B2,1", Or("P1,1", "P2,2", "P3,1"))
R4 = Neg("B1,1")
R5 = "B2,1"

kb_ejemplo = And(R1, R2, R3, R4, R5)

print("simbolos:", ", ".join(sorted(simbolos(kb_ejemplo))))
print("KB ⊨ ¬P1,2:", tt_entails(kb_ejemplo, Neg("P1,2")))
print("KB ⊨ ¬P2,2:", tt_entails(kb_ejemplo, Neg("P2,2")))


simbolos: B1,1, B2,1, P1,1, P1,2, P2,1, P2,2, P3,1
KB ⊨ ¬P1,2: True
KB ⊨ ¬P2,2: False


Los mismos dos resultados que en 1.4, pero ahora salidos de un algoritmo
general y no de una función escrita a medida del problema. Son siete símbolos,
así que la tabla tiene 2⁷ = 128 filas.


In [26]:
from itertools import product

syms = sorted(simbolos(kb_ejemplo))
todos = [dict(zip(syms, v)) for v in product([False, True], repeat=len(syms))]

print(f"modelos posibles:            {len(todos)}")
print(f"modelos donde la KB es cierta: {sum(pl_true(kb_ejemplo, m) for m in todos)}")


modelos posibles:            128
modelos donde la KB es cierta: 3


`tt_entails` es **sólido**, porque implementa directamente la definición de
implicación, y **completo**, porque siempre termina: los modelos son finitos.
Pero cuesta O(2ⁿ) en tiempo con *n* símbolos.

### 3.5 — El muro

Para la cueva entera son cuatro símbolos por casilla —el
pozo, el wumpus, la brisa y el hedor— sobre dieciséis casillas.


In [27]:
n_simbolos = 4 * 16
print(f"{n_simbolos} simbolos → {2 ** n_simbolos:.3g} modelos")


64 simbolos → 1.84e+19 modelos


Dieciocho trillones de modelos. 

Pero hay una salida más eficiente: **preguntar menos**. Dos observaciones alcanzan.

La primera: la KB solo habla de las casillas visitadas y de sus vecinas. Del
resto de la cueva no dice absolutamente nada, así que no hace falta
enumerarlas —y sobre ellas el agente no puede probar nada, que es la respuesta
correcta—. Llamamos a ese conjunto las casillas *en juego*.

La segunda: ninguna sentencia mezcla pozos con wumpus. Las brisas hablan de
pozos y los hedores hablan del wumpus, y la KB es la conjunción de dos partes
que no comparten ningún símbolo. Entonces se pueden preguntar por separado, y
en vez de 2^(2k) modelos son 2^k más 2^k. Van a ser **dos bases**, una por
mitad.

Con eso cada consulta pasa de dieciocho trillones de modelos a unos pocos
cientos.

#### 3.5.1 — Una base de conocimiento

Hasta acá veníamos armando sentencias sueltas. Con `tt_entails` escrito ya
podemos completar la clase que quedó declarada en 1.1: `TELL` agrega una
sentencia y `ASK` responde si algo se sigue de las que ya están.


In [28]:
class BaseProposicional(KnowledgeBase):
    """Una base de conocimiento proposicional, que responde consultas por
    verificacion de modelos."""

    def TELL(self, sentencia):
        # Una base de conocimiento es un *conjunto* de sentencias: decir dos
        # veces lo mismo no agrega nada.
        if sentencia not in self.sentences:
            self.sentences.append(sentencia)

    def ASK(self, query, symbols=None):
        # `symbols` es el atajo de 3.5: enumerar solo los simbolos que
        # importan, en vez de todos los que aparecen en la base.
        return tt_entails(self.sentencia(), query, symbols)

    def sentencia(self):
        """Toda la base como una sola sentencia: la conjuncion de sus partes."""
        return And(*self.sentences)


La misma consulta de 3.4, ahora contra un objeto en vez de contra una tupla
armada a mano:


In [29]:
kb = BaseProposicional()
for sentencia in (R1, R2, R3, R4, R5):
    kb.TELL(sentencia)

print("KB ⊨ ¬P1,2:", kb.ASK(Neg("P1,2")))
print("KB ⊨ ¬P2,2:", kb.ASK(Neg("P2,2")))


KB ⊨ ¬P1,2: True
KB ⊨ ¬P2,2: False


La base **solo crece**: TELL agrega, nunca saca. Es la monotonicidad de 1.7, y
es lo que le permite al agente ir acumulando percepciones sin tener que
revisar nunca nada de lo que ya concluyó.

### 3.6 — El agente

Es una versión simplificada de HYBRID-WUMPUS-AGENT.
"Híbrido" porque combina dos cosas que ya tenemos: la inferencia lógica de esta
unidad para saber **qué casillas son seguras**, y la búsqueda de la unidad 1
para saber **cómo llegar** hasta ellas.


In [30]:
from breadth_first_search import breadth_first_search
from node import failure, path_states
from problem import Problem


class RutaSegura(Problem):
    """Ir de una casilla a alguna de `GOALS` pisando solo las permitidas."""

    def __init__(self, INITIAL, GOALS, permitidas, n=4):
        super().__init__(INITIAL, None, GOALS=GOALS, permitidas=permitidas, n=n)

    def ACTIONS(self, state):
        return [v for v in vecinas(state, self.n) if v in self.permitidas]

    def RESULT(self, state, action):
        # La accion es la casilla vecina a la que se pasa.
        return action

    def IS_GOAL(self, state):
        return state in self.GOALS


El agente arranca con dos bases vacías y les hace TELL de su conocimiento de
fondo: en la casilla de partida no hay pozo ni wumpus. Después, una vez por
casilla nueva, traduce a sentencias lo que percibió.

De cada percepción salen dos cosas. Que sigue vivo parado ahí, así que esa
casilla está limpia. Y lo que dicen sus sensores: con brisa, alguna vecina
tiene pozo; sin brisa, ninguna. Esto último es la eliminación del bicondicional
de 1.7 aplicada a `Bx,y ⇔ (…)` junto con la percepción, y es lo que hace
desaparecer del problema los símbolos `B` y `S`.

El método `paso` es el `kb_agent` de 1.1, hecho concreto: le hace TELL a la
base de lo que percibió, le pregunta con ASK qué es seguro, y decide. Le falta
el tercer TELL de la figura 7.1, el de la acción ejecutada, porque nuestra base
no habla del tiempo y no tendría qué hacer con él. En 3.8 está lo que pasa
cuando sí.


In [31]:
def P(c):
    """El simbolo \"hay un pozo en la casilla c\"."""
    return f"P{c[0]},{c[1]}"


def W(c):
    """El simbolo \"hay un wumpus en la casilla c\"."""
    return f"W{c[0]},{c[1]}"


class AgenteWumpus:
    """HYBRID-WUMPUS-AGENT (figura 7.20), sin orientacion y sin flecha."""

    def __init__(self, n=4):
        self.n = n
        self.pos = (1, 1)
        self.percibido = {}      # casilla -> lo que percibio ahi
        self.plan = []           # las casillas que le falta caminar
        self.tiene_oro = False
        self.casilla_oro = None
        self.seguras = set()
        self.en_juego = []

        # Dos bases, porque los pozos y el wumpus no comparten ni un simbolo.
        self.pozos = BaseProposicional()
        self.wumpus = BaseProposicional()

        # El conocimiento de fondo: la casilla de partida es segura.
        self.ampliar((1, 1))
        self.pozos.TELL(Neg(P((1, 1))))
        self.wumpus.TELL(Neg(W((1, 1))))

    def ampliar(self, c):
        """Mete `c` y sus vecinas en juego, instanciando el wumpus unico."""
        for u in [c] + vecinas(c, self.n):
            if u not in self.en_juego:
                for v in self.en_juego:
                    # A lo sumo un wumpus: de cada par, alguna esta libre.
                    self.wumpus.TELL(Or(Neg(W(u)), Neg(W(v))))
                self.en_juego.append(u)

    def TELL(self, c, percept):
        """Traduce a sentencias lo que se percibio en la casilla `c`."""
        nueva = c not in self.percibido
        self.percibido[c] = percept
        if not nueva:
            return               # lo que se percibe en una casilla no cambia

        self.ampliar(c)

        # Esta parado ahi y sigue vivo: la casilla esta limpia.
        self.pozos.TELL(Neg(P(c)))
        self.wumpus.TELL(Neg(W(c)))

        alrededor = vecinas(c, self.n)
        if percept["brisa"]:
            self.pozos.TELL(Or(*[P(u) for u in alrededor]))
        else:
            for u in alrededor:
                self.pozos.TELL(Neg(P(u)))

        if percept["hedor"]:
            self.wumpus.TELL(Or(*[W(u) for u in alrededor]))
        else:
            for u in alrededor:
                self.wumpus.TELL(Neg(W(u)))

    def ASK(self):
        """Que casillas se pueden probar seguras con lo que hay en las bases."""
        pozos = [P(c) for c in self.en_juego]
        wumpus = [W(c) for c in self.en_juego]
        self.seguras = {c for c in self.en_juego
                        if self.pozos.ASK(Neg(P(c)), pozos)
                        and self.wumpus.ASK(Neg(W(c)), wumpus)}

    def plan_route(self, objetivos):
        """La ruta mas corta hasta algun objetivo, por casillas seguras."""
        objetivos = objetivos - {self.pos}
        if not objetivos:
            return []
        permitidas = self.seguras | set(self.percibido)
        nodo = breadth_first_search(
            RutaSegura(self.pos, objetivos, permitidas, self.n))
        return [] if nodo is failure else path_states(nodo)[1:]

    def paso(self, percept):
        self.TELL(self.pos, percept)
        self.ASK()

        if percept["brillo"]:
            self.casilla_oro = self.pos

        if percept["brillo"] and not self.tiene_oro:
            self.tiene_oro = True
            self.plan = self.plan_route({(1, 1)}) + ["Climb"]
            return "Grab"

        if not self.plan:
            sin_visitar = {c for c in self.seguras if c not in self.percibido}
            self.plan = self.plan_route(sin_visitar)

        if not self.plan:
            # No queda nada seguro por explorar: a casa.
            self.plan = self.plan_route({(1, 1)}) + ["Climb"]

        accion = self.plan.pop(0)
        if accion != "Climb":
            self.pos = accion
        return accion


Así queda la base de los pozos después de las dos primeras casillas del
recorrido, sin brisa en `[1,1]` y con brisa en `[2,1]`:


In [32]:
agente = AgenteWumpus()
agente.TELL((1, 1), mundo.PERCEPT((1, 1)))
agente.TELL((2, 1), mundo.PERCEPT((2, 1)))
agente.ASK()

for sentencia in agente.pozos.sentences:
    print(escribir(sentencia))

print()
print("casillas en juego:", len(agente.en_juego))
print("probadas seguras: ", sorted(agente.seguras))


¬P1,1
¬P2,1
¬P1,2
(P3,1 ∨ P1,1 ∨ P2,2)

casillas en juego: 5
probadas seguras:  [(1, 1), (1, 2), (2, 1)]


Las dos que ya pisó y `[1,2]`, que queda descartada por la **falta** de brisa y
de hedor en `[1,1]`. `[2,2]` y `[3,1]` no: con la brisa de `[2,1]`, el pozo
puede estar en cualquiera de las dos y no hay con qué decidir. Por ese lado el
agente está trabado, y le conviene volver sobre sus pasos y probar por `[1,2]`,
que es lo único seguro que le queda sin visitar.

Y el episodio: percibir, decidir, actuar, hasta salir de la cueva.


In [33]:
def episodio(mundo, agente, max_t=40):
    """Corre el agente hasta que sale de la cueva, se muere o se aburre."""
    camino = [agente.pos]
    for t in range(max_t):
        percept = mundo.PERCEPT(agente.pos, agente.tiene_oro)
        accion = agente.paso(percept)
        # Grab y Climb no mueven al agente: el camino no cambia.
        if accion not in ("Grab", "Climb"):
            camino.append(agente.pos)
        if accion == "Climb":
            final = "salió con el oro" if agente.tiene_oro else "salió sin nada"
            return t + 1, final, camino
        if mundo.es_mortal(agente.pos):
            return t + 1, f"murió en {agente.pos}", camino
    return max_t, "se quedó sin tiempo", camino


agente = AgenteWumpus()
pasos, final, camino = episodio(mundo, agente)

print(f"{final}, en {pasos} pasos")
print(" → ".join(f"[{x},{y}]" for x, y in camino))


salió con el oro, en 12 pasos
[1,1] → [2,1] → [1,1] → [1,2] → [2,2] → [3,2] → [2,2] → [2,3] → [2,2] → [1,2] → [1,1]


Sale con el oro. Y nunca pisó una casilla que no hubiera probado segura antes
de entrar: la garantía no viene del cuidado con que está escrito el agente,
viene de que la inferencia es sólida.

### 3.7 — La corrida, paso a paso


In [34]:
#| include: false
from html import escape

from IPython.display import HTML

from wumpus_web import Cueva


def ver(view, alto=780):
    """Mete la pagina que grabo el view adentro de esta celda."""
    return HTML(f'<div><iframe srcdoc="{escape(view.html(), quote=True)}" '
                f'style="width:100%; height:{alto}px; border:1px solid #dee2e6; '
                f'border-radius:.25rem" title="el agente paso a paso"></iframe></div>')


def saber(agente, n=4):
    """Lo que el agente tiene probado sobre cada casilla de la cueva."""
    pozos = [P(c) for c in agente.en_juego]
    wumpus = [W(c) for c in agente.en_juego]

    def veredicto(base, simbolo, libres):
        # tres respuestas posibles: probado que si, probado que no, o no se sabe
        if base.ASK(simbolo, libres):
            return "si"
        if base.ASK(Neg(simbolo), libres):
            return "no"
        return "?"

    d = {}
    for x in range(1, n + 1):
        for y in range(1, n + 1):
            c = (x, y)
            percept = agente.percibido.get(c, {})
            juega = c in agente.en_juego
            d[c] = {
                "visitado": c in agente.percibido,
                "ok": c in agente.seguras,
                "pozo": veredicto(agente.pozos, P(c), pozos) if juega else "?",
                "wumpus": veredicto(agente.wumpus, W(c), wumpus) if juega else "?",
                "brisa": percept.get("brisa", False),
                "hedor": percept.get("hedor", False),
                "oro": c == agente.casilla_oro,
                "tomado": agente.tiene_oro and c == agente.casilla_oro,
            }
    return d


def episodio_v(mundo, agente, view, max_t=40):
    """Igual que episodio, narrando cada paso."""
    for t in range(max_t):
        percept = mundo.PERCEPT(agente.pos, agente.tiene_oro)
        view.PERCEPT(t, agente.pos, percept)
        accion = agente.paso(percept)

        for c in agente.en_juego:
            if c not in agente.percibido:
                view.ASK(f"OK({c[0]},{c[1]})", c in agente.seguras)
        k = len(agente.en_juego)
        for nombre, base, simbolo in (("pozos", agente.pozos, P),
                                      ("wumpus", agente.wumpus, W)):
            libres = [simbolo(c) for c in agente.en_juego]
            consistentes = sum(pl_true(base.sentencia(), dict(zip(libres, v)))
                               for v in product([False, True], repeat=k))
            view.MODELOS(nombre, consistentes, 2 ** k)

        etiqueta = (accion if isinstance(accion, str)
                    else f"Forward → [{accion[0]},{accion[1]}]")
        view.PLAN([etiqueta] + [p if isinstance(p, str) else f"[{p[0]},{p[1]}]"
                                for p in agente.plan])
        view.STEP(saber(agente, mundo.n), etiqueta)

        if accion == "Climb":
            view.DONE("salió con el oro" if agente.tiene_oro else "salió",
                      f"En {t + 1} pasos, sin pisar nunca una casilla no probada.")
            return
        if mundo.es_mortal(agente.pos):
            view.DONE("muerto", f"Cayó en {agente.pos}.")
            return


recorrido = Cueva(mundo, "EL MUNDO DEL WUMPUS", "HYBRID-WUMPUS-AGENT",
                  "de [1,1] al oro y de vuelta")
episodio_v(mundo, AgenteWumpus(), recorrido)


In [35]:
#| echo: false
ver(recorrido)


Vale la pena mirar el cuadro del instante 3, cuando el agente vuelve de
`[2,1]` y entra a `[1,2]`. Ahí percibe hedor y no percibe brisa, y con eso
cierra tres inferencias:

- no hay brisa en `[1,2]`, así que ninguna vecina tiene pozo: `[2,2]` está
  limpia. Pero en `[2,1]` sí había brisa, y sus vecinas en duda eran `[2,2]` y
  `[3,1]`. Entonces **el pozo está en `[3,1]`**;
- hay hedor en `[1,2]`, así que el wumpus está en `[1,1]`, `[1,3]` o `[2,2]`.
  En `[1,1]` no está, porque es la casilla de arranque. En `[2,2]` tampoco,
  porque en `[2,1]` no había hedor. Entonces **el wumpus está en `[1,3]`**;
- sin pozo y sin wumpus, **`[2,2]` es segura**.
